In [51]:
import requests
from bs4 import BeautifulSoup
import time

urls = [
    # Wikipedia pages
    "https://en.wikipedia.org/wiki/Pakistan_Air_Force",
    "https://en.wikipedia.org/wiki/Pakistan_Air_Force_Academy",
    "https://en.wikipedia.org/wiki/No._1_Squadron_PAF",
    "https://en.wikipedia.org/wiki/JF-17_Thunder",
    "https://en.wikipedia.org/wiki/Pakistan_Air_Force_in_the_Indo-Pakistani_War_of_1965",
    "https://en.wikipedia.org/wiki/Pakistan_Air_Force_in_the_Indo-Pakistani_War_of_1971",

    # Official PAF website
    "https://www.paf.gov.pk/about-paf",
    "https://www.paf.gov.pk/history",
    "https://www.paf.gov.pk/air-bases",
]

all_text = ""

for url in urls:
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")

        # Unnecessary tags remove karo
        for tag in soup(["script", "style", "nav", "footer", "table"]):
            tag.decompose()

        text = soup.get_text(separator="\n", strip=True)
        all_text += f"\n\n=== SOURCE: {url} ===\n\n{text}"
        print(f"✅ {url} — {len(text)} chars")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error: {url} — {e}")

# Save karo
with open("paf_data.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print(f"\n✅ Total data: {len(all_text)} characters")

✅ https://en.wikipedia.org/wiki/Pakistan_Air_Force — 135708 chars
✅ https://en.wikipedia.org/wiki/Pakistan_Air_Force_Academy — 22174 chars
✅ https://en.wikipedia.org/wiki/No._1_Squadron_PAF — 4912 chars
✅ https://en.wikipedia.org/wiki/JF-17_Thunder — 128002 chars
✅ https://en.wikipedia.org/wiki/Pakistan_Air_Force_in_the_Indo-Pakistani_War_of_1965 — 1744 chars
✅ https://en.wikipedia.org/wiki/Pakistan_Air_Force_in_the_Indo-Pakistani_War_of_1971 — 1744 chars
❌ Error: https://www.paf.gov.pk/about-paf — HTTPSConnectionPool(host='www.paf.gov.pk', port=443): Max retries exceeded with url: /about-paf (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7adcf7c356d0>, 'Connection to www.paf.gov.pk timed out. (connect timeout=10)'))
❌ Error: https://www.paf.gov.pk/history — HTTPSConnectionPool(host='www.paf.gov.pk', port=443): Max retries exceeded with url: /history (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7adcf7c349e0>, 'Connection

In [52]:
import requests
from bs4 import BeautifulSoup

urls = [
    "https://en.wikipedia.org/wiki/Operation_Bunyan_ul_Marsoos",
    "https://en.wikipedia.org/wiki/2025_India%E2%80%93Pakistan_war",
    "https://en.wikipedia.org/wiki/Indo-Pakistani_War_of_2025",
]

all_text = ""
for url in urls:
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        all_text += f"\n\n=== {url} ===\n\n{text}"
        print(f"✅ {url} — {len(text)} chars")
    except Exception as e:
        print(f"❌ {e}")

# Purane data mein add karo
with open("paf_data.txt", "a", encoding="utf-8") as f:
    f.write(all_text)

print(f"✅ Naya data add ho gaya: {len(all_text)} chars")

✅ https://en.wikipedia.org/wiki/Operation_Bunyan_ul_Marsoos — 1569 chars
✅ https://en.wikipedia.org/wiki/2025_India%E2%80%93Pakistan_war — 132802 chars
✅ https://en.wikipedia.org/wiki/Indo-Pakistani_War_of_2025 — 132805 chars
✅ Naya data add ho gaya: 267386 chars


In [53]:
!pip install langchain langchain-community faiss-cpu sentence-transformers langchain-groq langchain-text-splitters streamlit pyngrok -q
print("✅ Done!")

✅ Done!


In [54]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("paf_data.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(documents)
print(f"✅ {len(chunks)} chunks ban gaye!")

✅ 677 chunks ban gaye!


In [55]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("⏳ Model load ho raha hai...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ Vector store ready!")

⏳ Model load ho raha hai...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store ready!


In [56]:
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")

with open("paf_app.py", "r") as f:
    content = f.read()

content = content.replace("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", groq_key)

with open("paf_app.py", "w") as f:
    f.write(content)

print("✅ Key set ho gayi!")

✅ Key set ho gayi!


In [57]:
# Cloudflare tunnel install karo
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ Cloudflared ready!")

cloudflared: Text file busy
✅ Cloudflared ready!


In [58]:
from google.colab import userdata
groq_key = userdata.get("GROQ_API_KEY")

app_code = '''import streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

st.set_page_config(page_title="PAF Assistant", page_icon="✈️")

st.markdown("""
<style>
    .stApp { background-color: #0a1628; color: white; }
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    .stButton > button {
        background: #1a3a6b;
        color: white;
        border: 1px solid #4a7abf;
        border-radius: 8px;
        width: 100%;
    }
</style>
""", unsafe_allow_html=True)

GROQ_KEY = "''' + groq_key + '''"

@st.cache_resource
def load_chain():
    loader = TextLoader("paf_data.txt", encoding="utf-8")
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 6, "fetch_k": 12})
    prompt = PromptTemplate(
        template="""You are an expert about Pakistan Air Force.
Give detailed answers using context below.
Chat History: {chat_history}
Context: {context}
Question: {question}
Answer:""",
        input_variables=["context", "question", "chat_history"]
    )
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2, api_key=GROQ_KEY)
    def format_docs(docs):
        return "\\n\\n".join(doc.page_content for doc in docs)
    return retriever, prompt, llm, format_docs

st.markdown("""
<div style="background:linear-gradient(135deg,#0a1628,#1a3a6b);
padding:20px;border-radius:12px;margin-bottom:20px;text-align:center;">
<h2 style="color:white;">✈️ Pakistan Air Force Assistant</h2>
<p style="color:#8ab4d4;">Ask anything about PAF history, aircraft, operations & more</p>
</div>
""", unsafe_allow_html=True)

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "bot", "content": "Assalam o Alaikum! Ask me anything about Pakistan Air Force!"}
    ]

for msg in st.session_state.messages:
    if msg["role"] == "bot":
        st.markdown('<div style="background:#1e2d4a;color:#e0e0e0;padding:10px 14px;border-radius:18px;margin:8px 60px 8px 8px;">✈️ ' + msg["content"] + '</div>', unsafe_allow_html=True)
    else:
        st.markdown('<div style="background:#1a3a6b;color:white;padding:10px 14px;border-radius:18px;margin:8px 8px 8px 60px;text-align:right;">' + msg["content"] + '</div>', unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

col1, col2, col3, col4 = st.columns(4)
with col1:
    if st.button("History"): st.session_state.quick = "Tell me about PAF history"
with col2:
    if st.button("Aircraft"): st.session_state.quick = "What aircraft does PAF use?"
with col3:
    if st.button("1965 War"): st.session_state.quick = "PAF role in 1965 war"
with col4:
    if st.button("Academy"): st.session_state.quick = "Tell me about PAF Academy"

question = st.chat_input("Ask about Pakistan Air Force...")

if "quick" in st.session_state:
    question = st.session_state.quick
    del st.session_state.quick

if question:
    st.session_state.messages.append({"role": "user", "content": question})
    chat_history = ""
    for msg in st.session_state.messages[-6:]:
        role = "User" if msg["role"] == "user" else "Assistant"
        chat_history += role + ": " + msg["content"] + "\\n"
    with st.spinner("Searching PAF records..."):
        retriever, prompt, llm, format_docs = load_chain()
        docs = retriever.invoke(question)
        context = format_docs(docs)
        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "question": question, "chat_history": chat_history})
    st.session_state.messages.append({"role": "bot", "content": answer})
    st.rerun()
'''

with open("paf_app.py", "w") as f:
    f.write(app_code)

print("✅ File ready!")

✅ File ready!


In [59]:
import subprocess, time

subprocess.run(["pkill", "-f", "streamlit"])
time.sleep(3)

subprocess.Popen([
    "streamlit", "run", "paf_app.py",
    "--server.port=8501",
    "--server.headless=true"
])

time.sleep(10)
print("✅ Restart ho gaya!")
print("🌐 Browser refresh karo!")

✅ Restart ho gaya!
🌐 Browser refresh karo!


In [60]:
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")

with open("paf_app.py", "r") as f:
    content = f.read()

# Purani key replace karo
import re
content = re.sub(r'GROQ_KEY = "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"', f'GROQ_KEY = "{groq_key}"', content)

with open("paf_app.py", "w") as f:
    f.write(content)

print(f"✅ Nai key set ho gayi: {groq_key[:8]}...")

✅ Nai key set ho gayi: gsk_YXd0...


In [61]:
import subprocess, time
subprocess.run(["pkill", "-f", "streamlit"])
time.sleep(3)
subprocess.Popen(["streamlit", "run", "paf_app.py", "--server.port=8501", "--server.headless=true"])
time.sleep(8)
print("✅ Browser refresh karo!")

✅ Browser refresh karo!


In [62]:
import subprocess
import threading
import time

# Streamlit restart
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(3)

# Streamlit start
subprocess.Popen([
    "streamlit", "run", "paf_app.py",
    "--server.port=8501",
    "--server.headless=true"
])
time.sleep(8)

# Naya Cloudflare tunnel
def run_tunnel():
    process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    for line in process.stderr:
        line = line.decode().strip()
        if "https://" in line and "trycloudflare.com" in line:
            for word in line.split():
                if word.startswith("https://") and "trycloudflare.com" in word:
                    print(f"\n✅ Naya Link:")
                    print(f"🌐 {word}")
                    break
            break

thread = threading.Thread(target=run_tunnel)
thread.start()
time.sleep(15)


✅ Naya Link:
🌐 https://locally-candy-adaptor-accurate.trycloudflare.com
